# RUL PREDICTING

Unlike the notebooks in the "learning_notebooks" folder - this file is supposed to be a polished final approach to the PREDICTING part of the RUL prediction problem, whilst the other notebook act as a small documentation of my learning process.

For this notebook, I decided to use FD004, meaning the dataset that has:
1. 6 unknown operating conditions for the turbines, which we will identify by clustering the data into 6 clusters, using k-means-clustering.
2. 2 fault types instead of just one.

In [ ]:
import joblib
import pandas as pd
import numpy as np
import features
import preprocessing

bundle = joblib.load("models.joblib")
path = "./CMAPSSData/test_FD004.txt"

# loading data
data = preprocessing.load_data(path, bundle=bundle)

# preprocessing data
data = preprocessing.apply_preprocessing(data, bundle)

# adding new features
new_features = features.get_new_features(data, bundle["sensors"])

# adding features to data
data = pd.concat([data, new_features], axis=1)

last_rows = data.groupby("engine_id").tail(1)

X = last_rows[bundle["model_columns"]]


Y1 = bundle["model_1"].predict(X).clip(min=0)
Y2 = bundle["model_2"].predict(X).clip(min=0)

predictions = last_rows[["engine_id", "cycle"]].copy()
predictions["predicted_RUL"] = (Y1 + Y2) / 2

predictions["actual_RUL"] = pd.read_csv("./CMAPSSData/RUL_FD004.txt", sep=r"\s+", header=None).iloc[:, 0].to_list()
predictions.to_csv("rul_comparison.csv", index=False)

In [19]:
## AI generated evaluation block:

from sklearn.metrics import mean_absolute_error, r2_score

actual_rul = pd.read_csv("./CMAPSSData/RUL_FD004.txt", sep=r"\s+", header=None).iloc[:, 0]

# Ensure motor oder
results = predictions.sort_values("engine_id").reset_index(drop=True).copy()

# Verify assignment
assert np.array_equal(
    results["engine_id"].to_numpy(),
    np.arange(1, len(actual_rul) + 1)
), "Motor-IDs dont fit RUL-file."

results["actual_RUL"] = actual_rul
results["error"] = results["predicted_RUL"] - results["actual_RUL"]

y_true = results["actual_RUL"]
y_pred = results["predicted_RUL"]

mae = mean_absolute_error(y_true, y_pred)
mse = (y_pred - y_true).mean()
rmse = np.sqrt(mse)
r2 = r2_score(y_true, y_pred)

print(f"Mean absolute error: {mae:.2f}")
print(f"Mean signed error: {mse:.2f}")
print(f"Root mean squared error: {rmse:.2f}")
print(f"R²:{r2:.3f}")

Mean absolute error: 15.03
Mean signed error: 0.49
Root mean squared error: 0.70
R²:0.837
